## 🔬 Project: Hyperspectral Image Classification - Indian Pines

### 📖 **Overview**
This project implements a **dual-branch deep learning architecture** for classifying the **Indian Pines** hyperspectral dataset. The model consists of:

- **🌿 Spectral Branch**: Transformer-based encoder that processes spectral information across 200 bands
- **🗺️ Spatial Branch**: Modified ResNet18 that extracts spatial features from 11×11 patches
- **🔗 Fusion Module**: Combines both branches using MLP + Bi-GRU + Attention Pooling

 ## 📚 Library Imports

This cell imports all necessary libraries for the Indian Pines HSI classification project:

🎯 **Purpose**: Centralise all imports for the entire project.

In [ ]:
# ==========================================
# 1. Standard Library Imports
# ==========================================
import time
from pathlib import Path

# ==========================================
# 2. Data Processing & Visualization
# ==========================================
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, classification_report
from IPython.display import clear_output
import tabulate

# ==========================================
# 3. Machine Learning
# ==========================================
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# ==========================================
# 4. Deep Learning - PyTorch
# ==========================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import torchvision.models as models
from torchsummary import summary


## 📊 Dataset Loading & Exploration

This cell loads the Indian Pines hyperspectral dataset and displays its key characteristics:

- **Dataset dimensions** (height, width, spectral bands)
- **Class distribution** (16 land cover classes)
- **Visualisation** of three representative spectral bands
- **Ground truth map** showing class labels spatially

> 📌 **Note**: Background pixels (class 0) are excluded from class distribution and classification tasks.

In [ ]:
# ==========================================
# Load Indian Pines dataset
# ==========================================
data = sio.loadmat('Indian_pines\Indian_pines_corrected.mat')['indian_pines_corrected']
gt = sio.loadmat('Indian_pines\Indian_pines_gt.mat')['indian_pines_gt']

# ==========================================
# Dataset dimensions and basic info
# ==========================================
h, w, bands = data.shape
num_pixels = h * w
num_classes = len(np.unique(gt)) - 1   # exclude background (class 0)

info_df = pd.DataFrame({
    "Property": [
        "Height",
        "Width",
        "Spectral Bands",
        "Total Pixels",
        "Number of Classes"
    ],
    "Value": [
        h,
        w,
        bands,
        num_pixels,
        num_classes
    ]
})

print("Dataset Information")
display(info_df)

# ==========================================
# Class distribution (excluding background)
# ==========================================
labels, counts = np.unique(gt, return_counts=True)

class_df = pd.DataFrame({
    "Class": labels,
    "Pixel Count": counts
})

class_df = class_df[class_df["Class"] != 0]   # remove background class

print("Class Distribution")
display(class_df)

# ==========================================
# Bar plot: class distribution
# ==========================================
plt.figure(figsize=(10,5))
plt.bar(class_df["Class"], class_df["Pixel Count"])
plt.xlabel("Class Label")
plt.ylabel("Pixel Count")
plt.title("Indian Pines Class Distribution")
plt.show()

# ==========================================
# Visualise three representative spectral bands
# ==========================================
bands_to_show = [10, 30, 60]

plt.figure(figsize=(12,4))
for i, b in enumerate(bands_to_show):
    plt.subplot(1, 3, i+1)
    plt.imshow(data[:, :, b], cmap='gray')
    plt.title(f'Band {b}')
    plt.axis('off')

plt.tight_layout()
plt.show()

# ==========================================
# Ground truth map (coloured by class)
# ==========================================
plt.figure(figsize=(5,5))
plt.imshow(gt, cmap='jet')
plt.title("Ground Truth Map")
plt.axis('off')
plt.show()

## ⚙️ Data Preprocessing & Loaders

This cell handles all data preparation steps:

- **Hyperparameters**: Patch size (11×11), batch size (64), and split ratios (10% train / 5% val / 85% test)
- **Normalisation**: Z‑score per spectral band for standardised input
- **Padding**: Enables proper patch extraction at image borders
- **Stratified Split**: Maintains class proportions across train/val/test sets
- **Dataset Class**: Extracts 11×11 patches with optional spatial augmentation (flips, rotations)
- **DataLoaders**: Batch generators for efficient training

> 🧠 **Augmentation**: Only applied to training data to improve generalisation.

In [ ]:
# ==========================================
# Hyperparameters
# ==========================================
D_MODEL = 96
PATCH_SIZE = 11
BATCH_SIZE = 64
RANDOM_STATE = 42

TRAIN_FRAC = 0.1          # fraction of labeled pixels for training
VAL_FRAC = 0.05           # fraction for validation

# ==========================================
# Z‑score normalisation per spectral band
# ==========================================
data = data.astype(np.float32)

mean = np.mean(data, axis=(0,1), keepdims=True)
std = np.std(data, axis=(0,1), keepdims=True) + 1e-6
data = (data - mean) / std

# ==========================================
# Pad the HSI cube for patch extraction
# ==========================================
h, w, bands = data.shape
margin = PATCH_SIZE // 2

padded_data = np.zeros(
    (h + 2*margin, w + 2*margin, bands),
    dtype=np.float32
)

padded_data[margin:margin+h, margin:margin+w, :] = data

# ==========================================
# Collect all labelled pixel positions
# ==========================================
indices = np.argwhere(gt > 0)
labels = gt[gt > 0] - 1          # shift labels to 0‑based

# ==========================================
# Train / validation / test split (stratified)
# ==========================================
train_idx, temp_idx, train_labels, temp_labels = train_test_split(
    np.arange(len(indices)),
    labels,
    train_size=TRAIN_FRAC,
    stratify=labels,
    random_state=RANDOM_STATE
)

val_relative_frac = VAL_FRAC / (1.0 - TRAIN_FRAC)

val_idx, test_idx = train_test_split(
    temp_idx,
    train_size=val_relative_frac,
    stratify=temp_labels,
    random_state=RANDOM_STATE
)

train_indices = indices[train_idx]
val_indices = indices[val_idx]
test_indices = indices[test_idx]

# ==========================================
# Print split sizes
# ==========================================
print("Total labeled pixels:", len(indices))
print(f"Train pixels ({TRAIN_FRAC*100}%):", len(train_indices))
print(f"Validation pixels ({VAL_FRAC*100}%):", len(val_indices))
print(f"Test pixels ({round((1 - TRAIN_FRAC - VAL_FRAC)*100)}%):", len(test_indices))

# ==========================================
# Dataset class with optional data augmentation
# ==========================================
class IndianPinesDataset(Dataset):

    def __init__(self, padded_data, indices, gt, patch_size, augment=False):
        self.data = padded_data
        self.indices = indices
        self.gt = gt
        self.patch_size = patch_size
        self.margin = patch_size // 2
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        x, y = self.indices[idx]

        x_pad = x + self.margin
        y_pad = y + self.margin

        # Extract the patch
        patch = self.data[
            x_pad-self.margin:x_pad+self.margin+1,
            y_pad-self.margin:y_pad+self.margin+1,
            :
        ]

        patch = torch.from_numpy(patch).permute(2,0,1).float()

        # Apply random spatial augmentations (only during training)
        if self.augment:
            # Random horizontal flip
            if torch.rand(1) > 0.5:
                patch = torch.flip(patch, [-1])
            # Random vertical flip
            if torch.rand(1) > 0.5:
                patch = torch.flip(patch, [-2])
            # Random 90° rotation
            if torch.rand(1) > 0.5:
                k = torch.randint(0, 4, (1,)).item()
                patch = torch.rot90(patch, k, [-2, -1])

        label = torch.tensor(self.gt[x,y]-1, dtype=torch.long)

        return patch, label

# ==========================================
# Create dataset instances
# ==========================================
train_dataset = IndianPinesDataset(
    padded_data,
    train_indices,
    gt,
    PATCH_SIZE,
    augment=True          # augment only training set
)

val_dataset = IndianPinesDataset(
    padded_data,
    val_indices,
    gt,
    PATCH_SIZE,
    augment=False
)

test_dataset = IndianPinesDataset(
    padded_data,
    test_indices,
    gt,
    PATCH_SIZE,
    augment=False
)

# ==========================================
# Create data loaders
# ==========================================
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# ==========================================
# Sanity check: inspect a batch
# ==========================================
x_batch, y_batch = next(iter(train_loader))

print("Batch patch shape:", x_batch.shape)
print("Batch labels shape:", y_batch.shape)

## 🗺️ Spatial Distribution of Train/Val/Test Pixels

This cell visualises how the labelled pixels are distributed across the image:

- **Ground Truth**: Complete class map (all 16 land cover types)
- **Train Pixels**: 10% of samples per class (stratified)
- **Validation Pixels**: 5% of samples per class
- **Test Pixels**: Remaining 85% of samples

> 🔍 **Observation**: The stratified split ensures each class appears in all three sets, preventing bias toward dominant classes.

In [ ]:
# ==========================================
# Visualise the spatial distribution of train / validation / test pixels
# ==========================================

# Create empty maps (same size as ground truth)
train_map = np.zeros_like(gt)
val_map = np.zeros_like(gt)
test_map = np.zeros_like(gt)

# Fill each map with the corresponding ground truth labels
for x, y in train_indices:
    train_map[x, y] = gt[x, y]

for x, y in val_indices:
    val_map[x, y] = gt[x, y]

for x, y in test_indices:
    test_map[x, y] = gt[x, y]

# Plot the four maps side by side
plt.figure(figsize=(16,4))

plt.subplot(1,4,1)
plt.title("Ground Truth")
plt.imshow(gt)
plt.axis("off")

plt.subplot(1,4,2)
plt.title("Train Pixels")
plt.imshow(train_map)
plt.axis("off")

plt.subplot(1,4,3)
plt.title("Validation Pixels")
plt.imshow(val_map)
plt.axis("off")

plt.subplot(1,4,4)
plt.title("Test Pixels")
plt.imshow(test_map)
plt.axis("off")

plt.show()

## 🧠 Spatial Branch: ResNet18 for HSI

This cell defines the **spatial feature extractor**:

### 🔍 **Attention Pooling Module**
- Learns **adaptive weights** over the 200 spectral bands
- Outputs a **single pooled vector** (B, D_model) for classification
- Dropout (0.4) prevents overfitting

### 🗺️ **SpatialResNet18 Architecture**
- **Modified ResNet18**: Adapted for single‑channel (1) input per band
- **Shared weights**: Same CNN processes all 200 bands independently
- **Truncated backbone**: Uses only `layer1`–`layer3` (reduces overfitting on small patches)
- **Projector**: Reduces channels from 256 → 96 (D_MODEL)
- **Two modes**:
  - **Phase 1**: Attention pooling + classifier → end‑to‑end training
  - **Phase 2**: Raw band‑wise features (B, 200, 96) → ready for fusion

> 💡 **Output dimension**: (Batch, 200 bands, 96 features) – matches spectral branch for easy fusion.

In [ ]:
# ==========================================
# Attention Pooling Module
# ==========================================
class AttentionPooling(nn.Module):
    def __init__(self, in_dim, hidden_dim=32):
        super().__init__()

        # Learnable attention weights over the sequence dimension
        self.attn = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        # x shape: (B, N, D) where N is sequence length
        weights = self.attn(x)               # (B, N, 1)
        weights = torch.softmax(weights, dim=1)
        pooled = torch.sum(weights * x, dim=1)  # (B, D)
        return pooled


# ==========================================
# Spatial Branch: Modified ResNet18 for HSI
# ==========================================
class SpatialResNet18(nn.Module):
    def __init__(self, out_dim=D_MODEL, num_classes=16, phase=1, pretrained_path=None):
        super().__init__()

        self.phase = phase

        # Load pretrained ResNet18 architecture (no weights)
        base_model = models.resnet18(weights=None)

        # Modify first conv layer for single-channel input
        base_model.conv1 = nn.Conv2d(
            1, 64, kernel_size=3, stride=1, padding=1, bias=False
        )
        base_model.maxpool = nn.Identity()

        # Use only first three residual blocks (layer1, layer2, layer3)
        self.backbone = nn.Sequential(
            base_model.conv1,
            base_model.bn1,
            base_model.relu,
            base_model.layer1,
            base_model.layer2,
            base_model.layer3,
        )

        # Projector: reduce channels from 256 to out_dim
        self.projector = nn.Sequential(
            nn.Conv2d(256, out_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=0.6),
        )

        self.pool2d = nn.AdaptiveAvgPool2d(1)

        # Phase 1: classification head
        if phase == 1:
            self.attn_pool = AttentionPooling(out_dim, hidden_dim=32)

            self.classifier = nn.Sequential(
                nn.Dropout(p=0.6),
                nn.Linear(out_dim, 32),
                nn.ReLU(inplace=True),
                nn.Dropout(p=0.4),
                nn.Linear(32, num_classes)
            )

        # Phase 2: load pretrained weights for feature extraction
        if phase == 2 and pretrained_path is not None:
            state = torch.load(pretrained_path, map_location="cpu")
            self.load_state_dict(state, strict=False)

    def forward(self, x):
        # Input shape: (B, Bands, H, W)
        B, Bands, H, W = x.shape

        # Process each band independently through the CNN backbone
        x = x.view(B * Bands, 1, H, W)
        x = self.backbone(x)
        x = self.projector(x)

        # Global average pooling and reshape back to band-wise features
        x = self.pool2d(x).squeeze(-1).squeeze(-1)
        x = x.view(B, Bands, -1)   # (B, 200, out_dim)

        # Phase 1: attention pooling + classification
        if self.phase == 1:
            x = self.attn_pool(x)      # (B, out_dim)
            return self.classifier(x)

        # Phase 2: return raw band-wise features for fusion
        if self.phase == 2:
            return x

## 🏋️ Spatial Branch Training Function

This cell defines the **training pipeline** for the spatial ResNet18 branch:

### ⚙️ **Key Features**
- **Differential Learning Rates**: Backbone (0.1×) vs. projection/classification heads (1×)
- **OneCycleLR**: Adaptive learning rate scheduling for faster convergence
- **Mixed Precision (AMP)**: Reduces memory usage and speeds up training
- **Gradient Clipping**: Prevents exploding gradients
- **Early Stopping**: Stops training after 30 epochs without improvement
- **Checkpoint Resume**: Saves/restores model state, optimizer, and scheduler

> 💡 **Output**: Trained model + DataFrame with training history (loss, accuracy, LR per epoch)

In [ ]:
# ==========================================
# Training function for Spatial ResNet18 branch
# ==========================================

def train_model(
    model,
    train_loader,
    val_loader,
    num_epochs=50,
    lr=1e-4,
    patience=30,
    device="cuda",
    checkpoint_path="checkpoint_spatial.pth",
    best_model_path="best_spatial.pth",
    class_weights=None,
    label_smoothing=0.0
):
    # Setup device
    device = torch.device(device if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    # Prepare class weights if provided
    if class_weights is not None:
        class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

    # Loss function with optional class weights and label smoothing
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)

    # Optimiser with differential learning rates
    optimizer = torch.optim.AdamW([
        {'params': model.backbone.parameters(), 'lr': lr * 0.1},   # lower LR for pretrained backbone
        {'params': model.projector.parameters()},
        {'params': model.attn_pool.parameters()},
        {'params': model.classifier.parameters()}
    ], lr=lr, weight_decay=5e-4)

    # OneCycleLR scheduler
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[lr * 0.1, lr, lr, lr],
        epochs=num_epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.2,
        div_factor=10,
        final_div_factor=1000
    )

    # Automatic Mixed Precision scaler
    scaler = GradScaler()

    start_epoch = 0
    best_val_acc = 0.0
    patience_counter = 0
    history = []

    # Resume from checkpoint if available
    try:
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        scaler.load_state_dict(ckpt["scaler_state"])
        start_epoch = ckpt["epoch"] + 1
        best_val_acc = ckpt.get("best_val_acc", 0.0)
        print(f"🔁 Resumed from epoch {start_epoch} | best acc = {best_val_acc:.2f}%")
    except FileNotFoundError:
        print("🆕 Starting fresh training...")

    # Training loop
    for epoch in range(start_epoch, num_epochs):
        t0 = time.time()
        model.train()
        train_loss, train_correct, total = 0, 0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", ncols=90)
        for imgs, labels in pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()

            # Forward pass with AMP
            with autocast(device_type="cuda", enabled=True):
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            # Backward pass with gradient scaling and clipping
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            train_loss += loss.item() * imgs.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        train_acc = 100 * train_correct / total
        train_loss /= total

        # Validation phase
        model.eval()
        val_correct, val_total, val_loss = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                with autocast(device_type="cuda", enabled=True):
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += labels.size(0)

        val_acc = 100 * val_correct / val_total
        val_loss /= val_total

        # Early stopping and best model saving
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"✅ New best accuracy: {best_val_acc:.2f}%")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n🛑 Early stopping triggered after {epoch+1} epochs")
                print(f"No improvement for {patience} epochs")
                break

        # Save checkpoint
        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "scaler_state": scaler.state_dict(),
                "best_val_acc": best_val_acc,
            },
            checkpoint_path,
        )

        # Logging
        epoch_time = time.time() - t0
        lr_now = optimizer.param_groups[1]["lr"]

        history.append(
            {
                "epoch": epoch + 1,
                "train_loss": round(train_loss, 4),
                "val_loss": round(val_loss, 4),
                "train_acc": round(train_acc, 2),
                "val_acc": round(val_acc, 2),
                "lr": f"{lr_now:.2e}",
                "time(s)": round(epoch_time, 1),
            }
        )

        # Display progress table
        clear_output(wait=True)
        df = pd.DataFrame(history)
        if tabulate:
            print(df.to_markdown(index=False))
        else:
            print(df.to_string(index=False))

    print(f"\n🏁 Training finished. Best Val Acc = {best_val_acc:.2f}%")
    return model, pd.DataFrame(history)

In [ ]:
# ==========================================
# Spatial Branch: Class Weight Calculation & Training
# ==========================================

print("Calculating class weights from train_loader...")
all_train_labels = []
for _, labels in train_loader:
    all_train_labels.extend(labels.cpu().numpy())

classes = np.arange(16)

# Compute balanced class weights and clip to avoid extreme values
raw_weights = compute_class_weight('balanced', classes=classes, y=all_train_labels)
class_weights = np.clip(raw_weights, 0.5, 3.0)   # prevents overfocus on rare classes
print("Original class weights:", raw_weights)
print("Clipped class weights:", class_weights)

# Initialise spatial branch (phase 1: classification)
model = SpatialResNet18(out_dim=D_MODEL, num_classes=16, phase=1)

print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

# Train the spatial branch
trained_model, log = train_model(
    model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=150,
    lr=3e-3,
    patience=30,
    device="cuda",
    checkpoint_path="checkpoint_spatial.pth",
    best_model_path="best_spatial.pth",
    class_weights=class_weights,
    label_smoothing=0.1
)

# ==========================================
# Evaluate spatial branch on validation set
# ==========================================
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to("cuda")
        outputs = model(imgs)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("\n" + "="*50)
print("PER-CLASS PERFORMANCE ON VALIDATION SET")
print("="*50)
print(classification_report(all_labels, all_preds, digits=3))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Validation Set')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# ==========================================
# Plot training history
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(log['epoch'], log['train_loss'], label='Train Loss')
axes[0].plot(log['epoch'], log['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(log['epoch'], log['train_acc'], label='Train Acc')
axes[1].plot(log['epoch'], log['val_acc'], label='Val Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# ==========================================
# Training summary
# ==========================================
print("\n" + "="*50)
print("TRAINING SUMMARY")
print("="*50)
print(f"Best validation OA: {log['val_acc'].max():.2f}%")
print(f"Best epoch: {log.loc[log['val_acc'].idxmax(), 'epoch']}")
print(f"Training stopped at epoch: {len(log)}")

## 🧪 Spatial Branch: Test Set Evaluation

This cell evaluates the trained spatial branch on the **held-out test set** (85% of data):

### 📊 **Evaluation Metrics**
- **Classification Report**: Precision, recall, F1‑score per class
- **Confusion Matrix**: Visualises correct vs. incorrect predictions

> 📌 **Key Insight**: Test metrics reveal how well the spatial branch generalises to unseen pixels. Compare with validation performance to detect overfitting.

In [ ]:
# =========================================================
# Test Evaluation
# =========================================================
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
# Load best model
model = SpatialResNet18(out_dim=D_MODEL, num_classes=16, phase=1).to("cuda")
model.load_state_dict(torch.load("best_spatial.pth"))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to("cuda")
        outputs = model(imgs)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("\n" + "="*50)
print("TEST SET PERFORMANCE")
print("="*50)
print(classification_report(all_labels, all_preds, digits=3))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Test Set')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

## 🌿 Spectral Branch: Transformer for HSI

This cell defines the **spectral feature extractor** using a Transformer encoder:

### 🔧 **Architecture Overview**

**1. Patch Embedding**
- Reshapes each 11×11 spatial patch into a vector (121 dims)
- Projects to `embed_dim=96` with LayerNorm

**2. Spectral Token & Position Embedding**
- Learnable `[SPECTRAL]` token prepended to the sequence (like BERT's [CLS])
- Position embeddings for all 200 bands + spectral token

**3. Transformer Encoder**
- **4 layers** with 12 attention heads each
- GELU activation + pre‑normalisation (norm_first)
- MLP dimension: 384 (4× embed_dim)

**4. Classification Head (Phase 1)**
- **Dual attention**: Spectral token (global) + learned band weights
- Simple classifier: Linear → GELU → Dropout → Linear(16)

**5. Feature Extraction (Phase 2)**
- Removes classification heads
- Returns per‑band features: `(B, 200, 96)` → ready for fusion

### spectral model

In [ ]:
# ==========================================
# Spectral Branch: Transformer-based Model for HSI
# ==========================================

class SpectralTransformer(nn.Module):
    def __init__(
        self,
        num_bands=200,
        patch_size=11,
        embed_dim=96,          # embedding dimension (matches spatial branch output)
        depth=4,               # number of transformer encoder layers
        heads=12,              # attention heads (embed_dim must be divisible by heads)
        mlp_dim=384,           # feed-forward network dimension
        num_classes=16,
        phase=1,               # 1 = classification, 2 = feature extraction for fusion
        dropout=0.2,           # dropout rate in transformer and classifier
        spectral_dropout=0.95, # keep probability for spectral dropout (5% dropout)
        pretrained_path=None
    ):
        super().__init__()

        self.phase = phase
        self.spectral_dropout_prob = spectral_dropout
        patch_dim = patch_size * patch_size

        # Project each spatial patch (within a band) to embedding dimension
        self.patch_embed = nn.Sequential(
            nn.Linear(patch_dim, embed_dim),
            nn.LayerNorm(embed_dim)
        )

        # Position embeddings for 200 spectral bands + learnable spectral token
        self.pos_embed = nn.Parameter(torch.randn(1, num_bands, embed_dim) * 0.02)
        self.spectral_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)

        # Transformer encoder with pre-normalisation and GELU activation
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=heads,
            dim_feedforward=mlp_dim,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)

        # Phase 1: classification head (spectral attention + classifier)
        if phase == 1:
            self.spectral_attention = nn.Sequential(
                nn.Linear(embed_dim, embed_dim // 2),
                nn.GELU(),
                nn.Linear(embed_dim // 2, 1)
            )

            self.classifier = nn.Sequential(
                nn.Linear(embed_dim, embed_dim // 2),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(embed_dim // 2, num_classes)
            )

        # Phase 2: feature extraction mode (no classification heads)
        elif phase == 2:
            self.spectral_attention = None
            self.classifier = None

    def forward(self, x):
        # Input shape: (B, C, H, W) where C = 200 spectral bands, H = W = 11
        B, C, H, W = x.shape

        # Apply spectral dropout (only during training and phase 1)
        if self.training and self.phase == 1:
            keep_prob = self.spectral_dropout_prob
            mask = torch.bernoulli(torch.full((B, C, 1, 1), keep_prob)).to(x.device)
            x = x * mask

        # Reshape: (B, C, H*W) then project to embedding dimension
        x = x.view(B, C, H * W)
        x = self.patch_embed(x)                # (B, C, embed_dim)

        # Add learnable spectral token (front of sequence)
        spectral_token = self.spectral_token.expand(B, -1, -1)
        x = torch.cat([spectral_token, x], dim=1)   # (B, C+1, embed_dim)

        # Add position embeddings (with zero embedding for the spectral token)
        pos_embed = torch.cat([torch.zeros(1, 1, self.pos_embed.shape[-1]).to(x.device),
                               self.pos_embed], dim=1)
        x = x + pos_embed

        # Pass through transformer encoder
        x = self.transformer(x)
        x = self.norm(x)

        # Phase 2: return band-wise features (exclude spectral token)
        if self.phase == 2:
            return x[:, 1:, :]                # (B, C, embed_dim)

        # Phase 1: classification
        # Global representation from spectral token
        spectral_features = x[:, 0, :]        # (B, embed_dim)

        # Learnable weighted sum over spectral bands
        attn_weights = self.spectral_attention(x[:, 1:, :])   # (B, C, 1)
        attn_weights = torch.softmax(attn_weights, dim=1)
        attended_features = torch.sum(attn_weights * x[:, 1:, :], dim=1)  # (B, embed_dim)

        # Combine both representations and classify
        combined_features = (spectral_features + attended_features) / 2
        out = self.classifier(combined_features)
        return out

## 🏋️ Spectral Branch Training Function

This cell defines the **training pipeline** for the spectral Transformer branch:

### 📊 **Training Configuration**
| Parameter | Value |
|-----------|-------|
| Max LR | 3e-3 |
| Weight decay | 1e-5 |
| Label smoothing | 0.05 |
| Dropout | 0.2 |
| Spectral dropout | 5% |

> 💡 **Output**: Trained model + DataFrame with complete training history

In [ ]:
# ==========================================
# Training function for Spectral Transformer branch
# ==========================================

def train_model(
    model,
    train_loader,
    val_loader,
    num_epochs=300,
    max_lr=3e-3,
    weight_decay=1e-5,
    label_smoothing=0.05,
    checkpoint_path="checkpoint_spectral.pth",
    best_model_path="best_model_spectral.pth",
):
    # Device selection (CUDA > MPS > CPU)
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        print("✅ Using MPS")
    else:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)

    # Compute class weights from training set (balanced + clipping)
    all_train_labels = []
    for _, labels in train_loader:
        all_train_labels.extend(labels.numpy())
    classes = np.arange(16)
    raw_weights = compute_class_weight('balanced', classes=classes, y=all_train_labels)
    class_weights = np.clip(raw_weights, 0.5, 3.0)
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

    print("Class weights (clipped):", class_weights.cpu().numpy())

    # Loss function with class weights and optional label smoothing
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)

    # AdamW optimiser with low weight decay
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=max_lr,
        weight_decay=weight_decay
    )

    # OneCycleLR scheduler for adaptive learning rate
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr,
        epochs=num_epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.3,
        anneal_strategy='cos',
        div_factor=25.0,
        final_div_factor=1000
    )

    start_epoch = 0
    best_val_acc = 0.0
    history = []

    # Resume from checkpoint if available
    try:
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        start_epoch = ckpt["epoch"] + 1
        best_val_acc = ckpt.get("best_val_acc", 0.0)
        print(f"🔄 Resumed from epoch {start_epoch}")
    except FileNotFoundError:
        print("🆕 Starting fresh training ...")

    # Training configuration summary
    print(f"\n📚 Training Configuration:")
    print(f"   Class weights: ENABLED")
    print(f"   Max LR: {max_lr}")
    print(f"   Weight decay: {weight_decay}")
    print(f"   Dropout: 0.2")
    print(f"   Spectral dropout: 5%")
    print(f"   Device: {device}")
    print("="*60 + "\n")

    # Training loop
    for epoch in range(start_epoch, num_epochs):
        t0 = time.time()
        model.train()
        train_loss, train_correct, total = 0, 0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for imgs, labels in pbar:
            imgs, labels = imgs.to(device), labels.to(device)

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            train_loss += loss.item() * imgs.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100 * train_correct / total:.2f}%'
            })

        train_acc = 100 * train_correct / total
        train_loss /= total

        # Validation phase
        model.eval()
        val_correct, val_total, val_loss = 0, 0, 0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * imgs.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += labels.size(0)

        val_acc = 100 * val_correct / val_total
        val_loss /= val_total

        # Save best model
        is_best = val_acc > best_val_acc
        if is_best:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_model_path)

        # Save checkpoint
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "best_val_acc": best_val_acc,
        }, checkpoint_path)

        # Logging
        epoch_time = time.time() - t0
        current_lr = optimizer.param_groups[0]['lr']

        history.append({
            "epoch": epoch + 1,
            "train_loss": round(train_loss, 4),
            "val_loss": round(val_loss, 4),
            "train_acc": round(train_acc, 2),
            "val_acc": round(val_acc, 2),
            "gap": round(train_acc - val_acc, 2),
            "lr": f"{current_lr:.2e}",
            "best_so_far": "✓" if is_best else ""
        })

        # Print detailed progress for every epoch
        print(f"\n{'='*70}")
        print(f"📊 EPOCH {epoch+1}/{num_epochs}")
        print(f"{'='*70}")
        print(f"   📈 TRAIN - Loss: {train_loss:.4f} | Accuracy: {train_acc:.2f}%")
        print(f"   📉 VALID - Loss: {val_loss:.4f} | Accuracy: {val_acc:.2f}%")
        print(f"   📊 Gap: {train_acc - val_acc:.1f}%")
        print(f"   🎯 Best Val Acc So Far: {best_val_acc:.2f}% {'⭐ NEW BEST! ⭐' if is_best else ''}")
        print(f"   ⚡ Learning Rate: {current_lr:.2e}")
        print(f"   ⏱️  Time: {epoch_time:.1f}s")

        # Show gradient statistics every 50 epochs
        if (epoch + 1) % 50 == 0:
            total_norm = 0
            for p in model.parameters():
                if p.grad is not None:
                    param_norm = p.grad.data.norm(2)
                    total_norm += param_norm.item() ** 2
            total_norm = total_norm ** 0.5
            print(f"   🔧 Gradient Norm: {total_norm:.4f}")

        print(f"{'='*70}\n")

    # Training completed summary
    print(f"\n{'='*70}")
    print(f"🏁 TRAINING COMPLETED!")
    print(f"{'='*70}")
    print(f"   Best Validation Accuracy: {best_val_acc:.2f}%")
    print(f"   Final Validation Accuracy: {val_acc:.2f}%")
    print(f"   Final Training Accuracy: {train_acc:.2f}%")
    print(f"   Final Gap: {train_acc - val_acc:.1f}%")
    print(f"{'='*70}\n")

    return model, pd.DataFrame(history)

## 🚀 Spectral Branch: Training Execution & Visualisation

This cell **initialises, trains, and evaluates** the spectral Transformer branch:

### 📦 **Model Initialisation**
- Creates `SpectralTransformer` with optimised hyperparameters
- Clears MPS cache (Apple Silicon) for clean start
- Displays model parameter count (trainable weights)

### 🏋️ **Training Execution**
- Runs `train_model()` with 300 epochs
- Class weights enabled (clipped to `[0.5, 3.0]`)
- Label smoothing = 0.05
- OneCycleLR scheduler with peak LR = 3e-3

### 📊 **Visualisation Outputs**
1. **Accuracy Curves**: Training vs. validation accuracy over epochs
2. **Overfitting Gap**: Difference between train and val accuracy
3. **Learning Rate Schedule**: OneCycleLR progression (log scale)

 💾 **Saved Files**:
 - `best_model_spectral.pth` → Best model weights
 - `checkpoint_model_spectral.pth` → Resumable checkpoint
 - `spectral_training.csv` → Complete training history
 - `spectral_training.png` → Training curves plot

In [ ]:
# ==========================================
# Spectral Branch: Model Initialisation & Training
# ==========================================

# Clear MPS cache (if using Apple Silicon)
if torch.backends.mps.is_available():
    torch.mps.empty_cache()
    print("✅ MPS cache cleared")

# Initialise spectral transformer model (phase 1: classification)
spectral_model = SpectralTransformer(
    num_bands=200,
    patch_size=11,
    embed_dim=96,          # embedding dimension
    depth=4,               # number of transformer layers
    heads=12,              # attention heads
    mlp_dim=384,           # feed-forward dimension
    num_classes=16,
    phase=1,
    dropout=0.2,           # dropout rate
    spectral_dropout=0.95  # keep 95% of spectral bands (5% dropout)
)

# Model summary
print("\n" + "="*60)
print(" SPECTRAL MODEL ")
print("="*60)
total_params = sum(p.numel() for p in spectral_model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"Model dimension: 64")
print(f"Transformer depth: 4")
print(f"Attention heads: 8")
print(f"MLP dimension: 384")
print(f"Dropout rate: 0.2")
print(f"Spectral dropout: 5%")
print(f"Weight decay: 1e-5")
print(f"Label smoothing: 0.05")
print(f"Class weights: ENABLED (clipped)")
print(f"Learning rate: 3e-3")
print("="*60 + "\n")

# Train the spectral branch
print("\n🔍 Training spectral branch (detailed progress every epoch)\n")
trained_model, training_log = train_model(
    spectral_model,
    train_loader,
    val_loader,
    num_epochs=300,
    max_lr=3e-3,
    weight_decay=1e-5,
    label_smoothing=0.05,
    checkpoint_path="checkpoint_model_spectral.pth",
    best_model_path="best_model_spectral.pth"
)

# Save training history
training_log.to_csv("spectral_training.csv", index=False)

# ==========================================
# Spectral Branch: Final Results & Visualisation
# ==========================================

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"Best validation accuracy: {training_log['val_acc'].max():.2f}%")
print(f"Final validation accuracy: {training_log['val_acc'].iloc[-1]:.2f}%")
print(f"Final training accuracy: {training_log['train_acc'].iloc[-1]:.2f}%")
print(f"Final overfitting gap: {training_log['gap'].iloc[-1]:.1f}%")
print(f"Total parameters: {total_params:,}")

# Plot training curves
plt.figure(figsize=(15, 5))

# Subplot 1: Accuracy curves
plt.subplot(1, 3, 1)
plt.plot(training_log['epoch'], training_log['train_acc'], label='Train Acc', linewidth=2)
plt.plot(training_log['epoch'], training_log['val_acc'], label='Val Acc', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training vs Validation Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 2: Overfitting gap
plt.subplot(1, 3, 2)
plt.plot(training_log['epoch'], training_log['gap'], label='Overfitting Gap', color='red', linewidth=2)
plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
plt.xlabel('Epoch')
plt.ylabel('Gap (%)')
plt.title('Train-Val Accuracy Gap')
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 3: Learning rate schedule
plt.subplot(1, 3, 3)
lr_values = [float(lr) for lr in training_log['lr']]
plt.plot(training_log['epoch'], lr_values, label='Learning Rate', color='green', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('OneCycleLR Schedule')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('spectral_training.png', dpi=150)
plt.show()

print("\n✅ Training curves saved to 'spectral_training.png'")

# Best epoch details
best_epoch = training_log['val_acc'].idxmax()
print(f"\n📈 Best Results at Epoch {best_epoch + 1}:")
print(f"   Validation Accuracy: {training_log.loc[best_epoch, 'val_acc']:.2f}%")
print(f"   Training Accuracy: {training_log.loc[best_epoch, 'train_acc']:.2f}%")
print(f"   Gap: {training_log.loc[best_epoch, 'gap']:.1f}%")
print(f"   Learning Rate: {training_log.loc[best_epoch, 'lr']}")

## 🧪 Spectral Branch: Test Set Evaluation

This cell evaluates the trained spectral Transformer on the **held-out test set** (85% of data):

### 📊 **Evaluation Metrics**
- **Overall Accuracy (OA)**: Percentage of correctly classified pixels
- **Average Accuracy (AA)**: Mean of per‑class accuracies (handles imbalance)
- **Weighted F1-Score**: Harmonic mean of precision & recall
- **Per‑Class Accuracy**: Individual performance for all 16 classes
- **Confusion Matrix**: Visualises misclassifications between classes
- **Classification Report**: Precision, recall, F1 per class

### 📁 **Output Files**
- `confusion_matrix_test.png` → Confusion matrix visualisation
- `test_results.json` → Metrics summary (OA, AA, F1)

In [ ]:
# ==========================================
# Test Set Evaluation for Spectral Branch
# ==========================================
def evaluate_model_on_test(model_path, model_class, test_loader, num_classes=16, device='mps'):
    """
    Load best trained model and evaluate on test set.
    
    Args:
        model_path: Path to saved model weights
        model_class: Model class (e.g., SpectralTransformer)
        test_loader: DataLoader for test set
        num_classes: Number of output classes
        device: Device to run evaluation on ('cuda', 'mps', or 'cpu')
    
    Returns:
        Dictionary containing evaluation metrics
    """
    # Setup device
    if device == 'mps' and torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    
    print(f"🔧 Using device: {device}")
    
    # Create model instance with same architecture as trained model
    model = model_class(
        num_bands=200,
        patch_size=11,
        embed_dim=96,
        depth=4,
        heads=12,
        mlp_dim=384,
        num_classes=num_classes,
        phase=1,
        dropout=0.2,
        spectral_dropout=0.95
    )
    
    # Load best model weights
    print(f"📂 Loading model from: {model_path}")
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    
    print("✅ Model loaded successfully!\n")
    
    # Evaluation loop
    all_preds = []
    all_labels = []
    test_loss = 0
    criterion = torch.nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            test_loss += loss.item() * imgs.size(0)
            
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Convert to numpy arrays for metric computation
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Calculate metrics
    overall_accuracy = accuracy_score(all_labels, all_preds) * 100
    avg_accuracy = np.mean([accuracy_score(all_labels[all_labels == c], all_preds[all_labels == c]) 
                            for c in range(num_classes) if np.any(all_labels == c)]) * 100
    f1 = f1_score(all_labels, all_preds, average='weighted') * 100
    
    # Per-class accuracies
    class_accuracies = {}
    for c in range(num_classes):
        mask = (all_labels == c)
        if np.any(mask):
            class_acc = accuracy_score(all_labels[mask], all_preds[mask]) * 100
            class_accuracies[c+1] = class_acc
    
    # Print test results summary
    print("="*70)
    print("📊 TEST SET RESULTS")
    print("="*70)
    print(f"🎯 Overall Accuracy (OA): {overall_accuracy:.2f}%")
    print(f"📈 Average Accuracy (AA): {avg_accuracy:.2f}%")
    print(f"🏆 Weighted F1-Score: {f1:.2f}%")
    print(f"📉 Test Loss: {test_loss / len(test_loader.dataset):.4f}")
    print("="*70)
    
    print("\n📋 Per-Class Accuracy:")
    print("-"*40)
    for class_id, acc in class_accuracies.items():
        print(f"   Class {class_id:2d}: {acc:5.2f}%")
    
    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    
    # Plot confusion matrix
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=[f'C{i+1}' for i in range(num_classes)],
                yticklabels=[f'C{i+1}' for i in range(num_classes)])
    plt.title('Confusion Matrix - Test Set', fontsize=16)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.tight_layout()
    plt.savefig('confusion_matrix_test.png', dpi=150)
    plt.show()
    
    # Detailed classification report
    print("\n📊 Detailed Classification Report:")
    print("="*70)
    target_names = [f'Class_{i+1}' for i in range(num_classes)]
    print(classification_report(all_labels, all_preds, target_names=target_names, digits=4))
    
    return {
        'overall_accuracy': overall_accuracy,
        'average_accuracy': avg_accuracy,
        'f1_score': f1,
        'confusion_matrix': cm,
        'predictions': all_preds,
        'labels': all_labels
    }


# ==========================================
# Run evaluation on test set
# ==========================================

# Path to the best saved model
BEST_MODEL_PATH = "best_model_spectral.pth"

# Evaluate spectral branch on test set
results = evaluate_model_on_test(
    model_path=BEST_MODEL_PATH,
    model_class=SpectralTransformer,
    test_loader=test_loader,
    num_classes=16,
    device='cuda'  # Change to 'mps' or 'cpu' as needed
)

# Save results to JSON file
import json
with open('test_results.json', 'w') as f:
    json.dump({
        'overall_accuracy': results['overall_accuracy'],
        'average_accuracy': results['average_accuracy'],
        'f1_score': results['f1_score']
    }, f, indent=2)

print("\n✅ Results saved to 'test_results.json'")

## 🔧 Device Setup & Phase 2 Model Loaders

This cell loads pre‑trained models in **feature extraction mode** (Phase 2):


### 📦 **Phase 2 Loaders**

**`load_spatial_phase2()`**
- Loads spatial ResNet18 trained weights
- Removes classification heads (`attn_pool`, `classifier`)
- Returns raw band‑wise features: `(B, 200, 96)`

**`load_spectral_phase2()`**
- Loads spectral Transformer trained weights
- Removes `spectral_attention` and `classifier` heads
- Returns per‑band features: `(B, 200, 96)`

### 🎯 **Purpose**
Phase 2 models serve as **frozen feature extractors** for the fusion module:
- No gradient computation during fusion training
- Both branches output **identical dimensions** (96) for seamless concatenation
- Preserves learned spectral/spatial representations

In [ ]:
# ==========================================
# Device Setup & Phase 2 Model Loaders
# ==========================================

# Select best available device (CUDA > MPS > CPU)
device = torch.device(
    "cuda" if torch.cuda.is_available() 
    else "mps" if torch.backends.mps.is_available() 
    else "cpu"
)

def load_spatial_phase2(checkpoint_path, out_dim=96, device=device):

    model = SpatialResNet18(out_dim=out_dim, phase=2)
    state = torch.load(checkpoint_path, map_location=device)
    # Phase 2 has no classifier or attn_pool layers; ignore missing keys
    model.load_state_dict(state, strict=False)
    model = model.to(device)
    return model

def load_spectral_phase2(checkpoint_path, embed_dim=96, depth=4, heads=12,
                         mlp_dim=384, dropout=0.2, spectral_dropout=0.95, device=device):

    model = SpectralTransformer(
        num_bands=200,
        patch_size=11,
        embed_dim=embed_dim,
        depth=depth,
        heads=heads,
        mlp_dim=mlp_dim,
        num_classes=16,
        phase=2,                      # Feature extraction mode
        dropout=dropout,
        spectral_dropout=spectral_dropout
    )
    state = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state, strict=False)
    model = model.to(device)
    return model

## 🔗 Fusion Module: Band‑wise MLP + Bi‑GRU + Attention Pooling

This cell defines the **fusion architecture** that combines spatial and spectral features:

### 🏗️ **Architecture Pipeline**

**1. Per‑Band MLP**
- Concatenates spatial `(B, 200, 96)` + spectral `(B, 200, 96)` → `(B, 200, 192)`
- Reduces to `fused_dim=64` with ReLU + dropout
- **Purpose**: Fuses complementary information within each spectral band

**2. Bidirectional GRU**
- Processes the 200 fused band vectors sequentially
- Hidden size = 32 → output `(B, 200, 64)` (bidirectional)
- **Purpose**: Models spectral dependencies across adjacent bands

**3. Attention Pooling**
- Learns adaptive weights over the 200 GRU outputs
- Pools to a single vector `(B, 64)`
- **Purpose**: Focuses on discriminative spectral regions

**4. Classifier Head**
- Two‑layer MLP: 64 → 32 → 16 classes
- Heavy dropout (0.6) prevents overfitting

### 📊 **Hyperparameters**
| Parameter | Value | Purpose |
|-----------|-------|---------|
| `fused_dim` | 64 | Compact band representation |
| `gru_hidden` | 32 | Captures spectral patterns |
| `dropout` | 0.6 | Strong regularisation |
| `attention_hidden` | 32 | Lightweight attention |

💡 **Output**: Final classification logits `(B, 16)` for Indian Pines classes

In [ ]:
# ==========================================
# Fusion Module: Band-wise MLP + Bi-GRU + Attention Pooling
# ==========================================

class TwoBranchFusion(nn.Module):
    def __init__(self, spat_dim, spec_dim, fused_dim=64,
                 gru_hidden=32, num_classes=16, dropout=0.6):
        super().__init__()

        # Step 1: Per-band MLP to fuse spatial and spectral features
        self.band_mlp = nn.Sequential(
            nn.Linear(spat_dim + spec_dim, fused_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )

        # Step 2: Bidirectional GRU to model sequential dependencies across bands
        self.gru = nn.GRU(fused_dim, gru_hidden,
                          bidirectional=True, batch_first=True)
        self.gru_dropout = nn.Dropout(dropout)

        # Step 3: Attention pooling over GRU outputs (band dimension)
        self.attn_pool = AttentionPooling(gru_hidden * 2, hidden_dim=32)

        # Step 4: Classifier head
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(gru_hidden * 2, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.7),
            nn.Linear(32, num_classes)
        )

    def forward(self, f_spat, f_spec):
        # Concatenate spatial and spectral features per band
        concat = torch.cat([f_spat, f_spec], dim=-1)          # (B, bands, spat_dim+spec_dim)
        
        # MLP to fuse features within each band
        fused_bands = self.band_mlp(concat)                   # (B, bands, fused_dim)
        
        # Bidirectional GRU to capture spectral dependencies
        gru_out, _ = self.gru(fused_bands)                    # (B, bands, 2*gru_hidden)
        gru_out = self.gru_dropout(gru_out)
        
        # Attention pooling over the band dimension
        pooled = self.attn_pool(gru_out)                      # (B, 2*gru_hidden)
        
        # Final classification
        return self.classifier(pooled)

## 🏋️ Fusion Module Training Function

This cell defines the **training pipeline** for the fusion module with frozen backbones:

### ⚙️ **Key Features**

**1. Freezed Feature Extractors**
- Spatial and spectral branches set to `eval()` mode
- No gradient computation → faster training, prevents catastrophic forgetting
- Only fusion parameters (MLP, GRU, attention, classifier) are trainable

**2. Class Weights & Regularisation**
- Balanced class weights clipped to `[0.5, 3.0]`
- Label smoothing = 0.1
- Strong L2 weight decay (`1e-2`) to prevent overfitting
- Heavy dropout (0.6) throughout fusion module

**3. Optimisation**
- AdamW optimiser with learning rate = `1e-3`
- OneCycleLR scheduler (warmup → high LR → annealing)
- Gradient clipping (max norm = 1.0)
- Automatic Mixed Precision (AMP) for efficiency

**4. Early Stopping**
- Patience = 25 epochs without validation improvement
- Saves best model based on validation accuracy
- Resumable checkpoints saved every epoch

### 📊 **Training Loop**
- Loads batches → extract features (no grad) → fusion forward → loss → backward
- Validation after each epoch
- Progress table with tabulate support (clean formatting)

In [ ]:
# ==========================================
# Fusion Module Training Function
# ==========================================

try:
    import tabulate
except ImportError:
    tabulate = None

import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from IPython.display import clear_output
import time

def train_fusion(spatial_model, spectral_model, fusion_model,
                 train_loader, val_loader,
                 num_epochs=150,
                 lr=1e-3,
                 weight_decay=1e-2,               # Strong L2 regularisation
                 patience=25,                     # Early stopping patience
                 checkpoint_path="checkpoint_fusion.pth",
                 best_model_path="best_fusion.pth"):
    """
    Train the fusion module while keeping spatial and spectral branches frozen.
    
    Args:
        spatial_model: Pre-trained spatial branch (phase 2, frozen)
        spectral_model: Pre-trained spectral branch (phase 2, frozen)
        fusion_model: Fusion module to train
        train_loader, val_loader: Data loaders
        num_epochs: Maximum number of training epochs
        lr: Learning rate
        weight_decay: L2 regularisation strength
        patience: Early stopping patience (epochs without improvement)
        checkpoint_path: Path to save training checkpoints
        best_model_path: Path to save best model weights
    
    Returns:
        Trained fusion model and training history DataFrame
    """
    
    device = next(spatial_model.parameters()).device
    fusion_model = fusion_model.to(device)

    # Freeze both feature extractors (spatial and spectral branches)
    for param in spatial_model.parameters():
        param.requires_grad = False
    for param in spectral_model.parameters():
        param.requires_grad = False
    spatial_model.eval()
    spectral_model.eval()

    # Compute class weights from training set (balanced + clipping)
    all_train_labels = []
    for _, labels in train_loader:
        all_train_labels.extend(labels.numpy())
    raw_weights = compute_class_weight('balanced', classes=np.arange(16), y=all_train_labels)
    class_weights = np.clip(raw_weights, 0.5, 3.0)
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

    # Loss function with class weights and label smoothing
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    
    # Optimiser with strong weight decay
    optimizer = torch.optim.AdamW(fusion_model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # OneCycleLR scheduler
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr,
        epochs=num_epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.3, anneal_strategy='cos'
    )
    
    # Automatic Mixed Precision scaler
    scaler = GradScaler()

    start_epoch = 0
    best_val_acc = 0.0
    patience_counter = 0
    history = []

    # Resume from checkpoint if available
    try:
        ckpt = torch.load(checkpoint_path, map_location=device)
        fusion_model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        scaler.load_state_dict(ckpt["scaler_state"])
        start_epoch = ckpt["epoch"] + 1
        best_val_acc = ckpt.get("best_val_acc", 0.0)
        print(f"🔁 Resumed from epoch {start_epoch} | best acc = {best_val_acc:.2f}%")
    except FileNotFoundError:
        print("🆕 Starting fresh fusion training...")

    # Training loop
    for epoch in range(start_epoch, num_epochs):
        t0 = time.time()
        fusion_model.train()
        train_loss, train_correct, total = 0, 0, 0

        pbar = tqdm(train_loader, desc=f"Fusion Epoch {epoch+1}/{num_epochs}", ncols=90)
        for imgs, labels in pbar:
            imgs, labels = imgs.to(device), labels.to(device)

            # Extract features from frozen branches (no gradients)
            with torch.no_grad():
                f_spat = spatial_model(imgs)   # (B, bands, spat_dim)
                f_spec = spectral_model(imgs)  # (B, bands, spec_dim)

            # Forward pass through fusion module with AMP
            with autocast(device_type="cuda", enabled=True):
                outputs = fusion_model(f_spat, f_spec)
                loss = criterion(outputs, labels)

            # Backward pass with gradient scaling and clipping
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(fusion_model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            train_loss += loss.item() * imgs.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        train_acc = 100 * train_correct / total
        train_loss /= total

        # Validation phase
        fusion_model.eval()
        val_correct, val_total, val_loss = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                f_spat = spatial_model(imgs)
                f_spec = spectral_model(imgs)
                with autocast(device_type="cuda", enabled=True):
                    outputs = fusion_model(f_spat, f_spec)
                    loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += labels.size(0)
        val_acc = 100 * val_correct / val_total
        val_loss /= val_total

        # Early stopping and best model saving
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(fusion_model.state_dict(), best_model_path)
            print(f"✅ New best accuracy: {best_val_acc:.2f}%")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n🛑 Early stopping triggered after {epoch+1} epochs")
                print(f"No improvement for {patience} epochs")
                break

        # Save checkpoint
        torch.save({
            "epoch": epoch,
            "model_state": fusion_model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "best_val_acc": best_val_acc,
        }, checkpoint_path)

        # Logging
        epoch_time = time.time() - t0
        lr_now = optimizer.param_groups[0]["lr"]

        history.append({
            "epoch": epoch + 1,
            "train_loss": round(train_loss, 4),
            "val_loss": round(val_loss, 4),
            "train_acc": round(train_acc, 2),
            "val_acc": round(val_acc, 2),
            "lr": f"{lr_now:.2e}",
            "time(s)": round(epoch_time, 1),
        })

        # Display progress table
        clear_output(wait=True)
        df = pd.DataFrame(history)
        if tabulate:
            print(df.to_markdown(index=False))
        else:
            print(df.to_string(index=False))

    print(f"\n🏁 Fusion training finished. Best Val Acc = {best_val_acc:.2f}%")
    return fusion_model, pd.DataFrame(history)

## 🚀 Fusion Model: Training Execution & Validation

This cell **initialises, trains, and validates** the fusion module:

### 📦 **Setup & Initialisation**
- **Device detection**: CUDA → MPS → CPU with cache clearing
- **Load phase 2 models**: Pre-trained spatial & spectral feature extractors
- **Freeze backbones**: No gradient computation (memory efficient)
- **Fusion module**: 2‑branch fusion with heavy regularisation (dropout=0.5)

### 🏋️ **Training Configuration**
| Parameter | Value | Purpose |
|-----------|-------|---------|
| Learning rate | 5e-4 | Lower LR for stability |
| Weight decay | 5e-2 | Strong L2 regularisation |
| Patience | 15 epochs | Early stopping tolerance |
| Fused dim | 128 | Band representation size |
| GRU hidden | 64 | Bidirectional → 128 output |

### 📊 **Validation Evaluation**
- Loads best fusion model (`best_fusion.pth`)
- Runs inference on validation set
- Displays:
  - **Classification report** (precision, recall, F1 per class)
  - **Confusion matrix heatmap** (visualises misclassifications)

### 📈 **Training Visualisation**
- Loss curves (train vs. validation)
- Accuracy curves (train vs. validation)
- Helps detect overfitting / underfitting

💾 **Saved Files**:
- `best_fusion.pth` → Best fusion weights
- `checkpoint_fusion.pth` → Resumable checkpoint
- `fusion_training_log.csv` → Complete training history

In [ ]:
# ==========================================
# Fusion Model: Initialisation & Training
# ==========================================

# Device setup with cache clearing
def setup_device():
    """Select best available device and clear its cache."""
    if torch.cuda.is_available():
        device = torch.device("cuda")
        torch.cuda.empty_cache()
        print("✅ Using CUDA (GPU)")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        torch.mps.empty_cache()
        print("✅ Using MPS (Apple Silicon)")
    else:
        device = torch.device("cpu")
        print("⚠️ Using CPU")
    return device

device = setup_device()

# Paths to pre-trained phase 2 feature extractors
spatial_best_path = "best_spatial.pth"
spectral_best_path = "best_model_spectral.pth"

# Load phase 2 models (feature extractors only)
spatial_feat = load_spatial_phase2(spatial_best_path, out_dim=96, device=device)
spectral_feat = load_spectral_phase2(spectral_best_path, embed_dim=96, depth=4,
                                     heads=12, mlp_dim=384, dropout=0.2,
                                     spectral_dropout=0.95, device=device)

# Freeze both feature extractors
for p in spatial_feat.parameters(): 
    p.requires_grad = False
for p in spectral_feat.parameters(): 
    p.requires_grad = False

# Initialise fusion module with higher dropout for regularisation
fusion_model = TwoBranchFusion(spat_dim=96, spec_dim=96, fused_dim=128,
                               gru_hidden=64, num_classes=16, dropout=0.5)

print(f"\nFusion model parameters: {sum(p.numel() for p in fusion_model.parameters()):,}")

# Train fusion module (early stopping will trigger when validation plateaus)
trained_fusion, fusion_log = train_fusion(
    spatial_feat, spectral_feat, fusion_model,
    train_loader, val_loader,
    num_epochs=150,
    lr=5e-4,                   # Lower peak learning rate for stability
    weight_decay=5e-2,         # Strong L2 regularisation
    patience=15,               # Wait 15 epochs before early stopping
    checkpoint_path="checkpoint_fusion.pth",
    best_model_path="best_fusion.pth"
)

# Save training history
fusion_log.to_csv("fusion_training_log.csv", index=False)

# ==========================================
# Plot Fusion Training Curves
# ==========================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(fusion_log['epoch'], fusion_log['train_loss'], label='Train')
ax1.plot(fusion_log['epoch'], fusion_log['val_loss'], label='Val')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(fusion_log['epoch'], fusion_log['train_acc'], label='Train')
ax2.plot(fusion_log['epoch'], fusion_log['val_acc'], label='Val')
ax2.set_title('Accuracy')
ax2.legend()
ax2.grid(True)

plt.show()

# ==========================================
# Evaluate Fusion Model on Validation Set
# ==========================================

# Load best saved fusion model weights
fusion_model.load_state_dict(torch.load("best_fusion.pth"))
fusion_model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        f_spat = spatial_feat(imgs)
        f_spec = spectral_feat(imgs)
        outputs = fusion_model(f_spat, f_spec)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

# Classification report
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(all_labels, all_preds, digits=3))

# Confusion matrix heatmap
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
import seaborn as sns
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Validation Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

## 🧪 Fusion Model: Test Set Evaluation

This cell evaluates the **trained fusion model** on the held-out test set (85% of data):

### 📊 **Evaluation Metrics**

| Metric | Description |
|--------|-------------|
| **Overall Accuracy (OA)** | Percentage of correctly classified test pixels |
| **Average Accuracy (AA)** | Mean of per‑class accuracies (handles class imbalance) |
| **Weighted F1-Score** | Harmonic mean of precision & recall (weighted by class support) |
| **Per‑Class Accuracy** | Individual performance for all 16 land cover classes |
| **Confusion Matrix** | Visualises misclassifications between classes |
| **Classification Report** | Precision, recall, F1 per class (3‑digit precision) |

In [ ]:
# =========================================================
# Test Set Evaluation for Fusion Model
# =========================================================

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import seaborn as sns
import numpy as np

def test_fusion(spatial_model, spectral_model, fusion_model, test_loader, 
                best_model_path="best_fusion.pth", num_classes=16):
    """
    Evaluate trained fusion model on test set.
    
    Args:
        spatial_model: Pre-trained spatial branch (phase 2, frozen)
        spectral_model: Pre-trained spectral branch (phase 2, frozen)
        fusion_model: Fusion module architecture (weights will be loaded)
        test_loader: DataLoader for test set
        best_model_path: Path to saved best fusion model weights
        num_classes: Number of output classes
    
    Returns:
        overall_accuracy, average_accuracy, f1_score, confusion_matrix
    """
    
    device = next(spatial_model.parameters()).device

    # Load best checkpoint weights
    fusion_model.load_state_dict(torch.load(best_model_path, map_location=device))
    fusion_model.to(device)
    fusion_model.eval()

    # Collect predictions and labels
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            
            # Extract features from both branches
            f_spat = spatial_model(imgs)
            f_spec = spectral_model(imgs)
            
            # Forward through fusion module
            outputs = fusion_model(f_spat, f_spec)
            _, pred = outputs.max(1)
            
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Compute metrics
    overall_acc = accuracy_score(all_labels, all_preds) * 100
    
    # Average accuracy (AA): mean of per-class accuracies
    avg_acc = np.mean([accuracy_score(all_labels[all_labels==c], all_preds[all_labels==c])
                       for c in range(num_classes) if np.any(all_labels==c)]) * 100
    
    f1 = f1_score(all_labels, all_preds, average='weighted') * 100
    cm = confusion_matrix(all_labels, all_preds)

    # Print test results
    print("\n" + "="*70)
    print("📊 TEST SET RESULTS (Fusion Model)")
    print("="*70)
    print(f"🎯 Overall Accuracy (OA): {overall_acc:.2f}%")
    print(f"📈 Average Accuracy (AA): {avg_acc:.2f}%")
    print(f"🏆 Weighted F1-Score:     {f1:.2f}%")
    print("="*70)

    # Per-class accuracy breakdown
    print("\n📋 Per‑Class Test Accuracy:")
    for c in range(num_classes):
        mask = (all_labels == c)
        if np.any(mask):
            acc_c = accuracy_score(all_labels[mask], all_preds[mask]) * 100
            print(f"   Class {c+1:2d}: {acc_c:5.2f}%")

    # Detailed classification report
    print("\n" + classification_report(all_labels, all_preds, digits=3))

    # Plot confusion matrix
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=[f'C{i+1}' for i in range(num_classes)],
                yticklabels=[f'C{i+1}' for i in range(num_classes)])
    plt.title('Confusion Matrix – Fusion Model (Test Set)')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.show()

    return overall_acc, avg_acc, f1, cm

# =========================================================
# Run Test Evaluation
# =========================================================

# Note: spatial_feat, spectral_feat, trained_fusion, and test_loader must be defined
# trained_fusion provides the architecture; weights are loaded from best_fusion.pth

test_acc, test_aa, test_f1, cm = test_fusion(
    spatial_feat,           # Pre-trained spatial feature extractor
    spectral_feat,          # Pre-trained spectral feature extractor
    trained_fusion,         # Fusion module architecture
    test_loader,            # Test set data loader
    best_model_path="best_fusion.pth"
)